# UJIIndoorLoc trên Google Colab

Notebook này tải mã từ một nhánh GitHub, tự liệt kê các model và cách xử lý RSS có trong nhánh đó, tải UJIIndoorLoc rồi chạy thí nghiệm. **Không lưu GitHub token trong notebook.** Bật GPU trước khi chạy BSRBF-KAN. Official test chỉ dùng để báo cáo sau khi chọn model bằng internal validation.


In [ ]:
import io
import zipfile
from pathlib import Path
import requests
import subprocess
import shutil
from google.colab import files

REPO = "VietUniverse/ips-wifi-research"
BRANCH = "main"  # đổi thành feature/data-robustness hoặc feature/kan-mixer để thử PR
root = Path("/content/repo_source")

try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None

if token:
    url = f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}"
    response = requests.get(url, headers={"Authorization": f"Bearer {token}",
                                         "Accept": "application/vnd.github+json"}, timeout=120)
    response.raise_for_status()
    repo_zip = response.content
    del token
else:
    print("Không tìm thấy Colab Secret GITHUB_TOKEN. Upload ZIP của nhánh GitHub bạn muốn chạy.")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy upload đúng một file ZIP của repo")
    repo_zip = next(iter(uploaded.values()))

if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(repo_zip)) as z:
    z.extractall(root)
project_files = list(root.rglob("pyproject.toml"))
if len(project_files) != 1:
    raise RuntimeError(f"Cần đúng một pyproject.toml; tìm thấy {len(project_files)}")
project = project_files[0].parent
print("Project:", project)
subprocess.run(["python", "-m", "pip", "install", "-e", str(project)], check=True)


In [ ]:
import shutil
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name}")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


## Chọn model và tiền xử lý

Danh sách dưới đây đọc trực tiếp từ các file plugin trong nhánh đã tải, không cần sửa notebook. Sau khi thành viên push file mới, đổi `BRANCH` ở ô đầu rồi chạy lại ô tải repo và ô chọn này. Chọn preset `mlp_smoke.json` để thử luồng trước. Preset Mixer chỉ chạy với `bsrbf_kan` và `rss_standard` (520 AP). Một cấu hình đầy đủ có thể mất nhiều giờ.


In [ ]:
import ipywidgets as widgets
from IPython.display import display
import json

options = json.loads(subprocess.check_output(
    ["python", "-m", "ips.experiments.cli", "--list-options"], text=True))
presets = sorted(str(p.relative_to(project)) for p in (project / "configs").glob("*.json"))
model_choice = widgets.Dropdown(options=options["models"], description="Model:", value="mlp" if "mlp" in options["models"] else None)
preprocess_choice = widgets.Dropdown(options=options["preprocessors"], description="RSS:", value="rss_standard" if "rss_standard" in options["preprocessors"] else None)
preset_choice = widgets.Dropdown(options=presets, description="Preset:", value="configs/mlp_smoke.json" if "configs/mlp_smoke.json" in presets else None)
preprocess_params_choice = widgets.Text(value="{}", description="RSS params:", placeholder='{"min_observations": 20}')
display(model_choice, preprocess_choice, preprocess_params_choice, preset_choice)
print("Models:", options["models"], "| Preprocessors:", options["preprocessors"])


In [ ]:
import torch
from datetime import datetime, timezone

RUN_NAME = datetime.now(timezone.utc).strftime("run_%Y%m%d_%H%M%S")
run_dir = Path("/content/runs") / RUN_NAME
selected_model = model_choice.value
selected_preprocess = preprocess_choice.value
selected_preset = preset_choice.value
preprocess_params = json.loads(preprocess_params_choice.value)
if not isinstance(preprocess_params, dict):
    raise ValueError("RSS params phải là JSON object, ví dụ {}")
if not all((selected_model, selected_preprocess, selected_preset)):
    raise ValueError("Hãy chọn model, RSS và preset")
settings = json.loads((project / selected_preset).read_text())
if settings.get("with_mixer") and (selected_model != "bsrbf_kan" or selected_preprocess != "rss_standard"):
    raise ValueError("Preset Mixer cần model bsrbf_kan và RSS rss_standard")
print("GPU available:", torch.cuda.is_available())
if selected_model == "bsrbf_kan" and not torch.cuda.is_available():
    print("Cảnh báo: BSRBF-KAN trên CPU có thể rất chậm.")
subprocess.run(["python", "-m", "ips.experiments.cli", "--config", str(project / selected_preset),
                "--model", selected_model, "--preprocess", selected_preprocess,
                "--preprocess-params", json.dumps(preprocess_params),
                "--data-dir", str(data_dir), "--output-dir", str(run_dir)], check=True)
print((run_dir / "summary.json").read_text())


In [ ]:
archive = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive)
files.download(archive)
